In [ ]:
from pathlib import Path
import pickle
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import t, ttest_ind

PROJECT_DIR = Path("/Users/zhouxinyu/Downloads/lpp_isc_matched_average_subjects")
ISC_DIR = PROJECT_DIR / "isc_maps"
KEEP_NS = {10, 12, 14}
LANG_ORDER = ["EN", "CN", "FR"]
LANG_COLORS = {
    "EN": "#68e368",
    "CN": "#f06666",
    "FR": "#51a6e2",
}
CONDITION_COLORS = {
    "top": "#4C9AFF",
    "bot": "#FF9F40",
    "rand": "#9E9E9E",
}
POINT_COLORS = {
    "top": "#1F5FBF",
    "bot": "#C76A00",
    "rand": "#616161",
}

RANDOM_SUMMARY_FILES = [
    ISC_DIR / "isc_en_random_n10,12,14_30iter_seed1.pkl",
    ISC_DIR / "isc_cn_random_n10,12,14_30iter_seed1.pkl",
    ISC_DIR / "isc_fr_random_n10,12,14_30iter_seed1.pkl",
]
TOPBOTTOM_FILES = {
    "EN": {
        "topbottom": ISC_DIR / "isc_en_topbottom_n4,8,12,16,20,24_10iter_seed1.pkl",
        "random": ISC_DIR / "isc_en_random_n4,8,12,16,20,24_10iter_seed1.pkl",
    },
    "CN": {
        "topbottom": ISC_DIR / "isc_cn_topbottom_n4,8,12,16,20,24_10iter_seed1.pkl",
        "random": ISC_DIR / "isc_cn_random_n4,8,12,16,20,24_10iter_seed1.pkl",
    },
}


def seed_from_path(path: Path) -> int:
    match = re.search(r"seed(\d+)", path.stem)
    if match is None:
        raise ValueError(f"Could not parse seed from {path.name}")
    return int(match.group(1))


def language_from_path(path: Path) -> str | None:
    match = re.search(r"isc_([a-z]{2})_", path.stem, flags=re.IGNORECASE)
    if match is None:
        return None
    return match.group(1).upper()


def mean_isc(values) -> float:
    return float(np.nanmean(np.asarray(values, dtype=float)))


def load_pickle(path: Path):
    with path.open("rb") as f:
        return pickle.load(f)


def require_files(paths):
    missing = [Path(path) for path in paths if not Path(path).exists()]
    if missing:
        missing_text = "\n".join(str(path) for path in missing)
        raise FileNotFoundError(f"Missing ISC pickle file(s):\n{missing_text}")


def get_case_insensitive(mapping: dict, key: str):
    key = key.lower()
    for current_key, value in mapping.items():
        if str(current_key).lower() == key:
            return value
    raise KeyError(key)


def iter_random_blocks(path: Path, result: dict):
    if not isinstance(result, dict) or not result:
        raise ValueError(f"Random ISC result is empty or invalid: {path}")

    first_key = next(iter(result))
    if isinstance(first_key, (int, np.integer)) or str(first_key).isdigit():
        language = language_from_path(path)
        if language is None:
            raise ValueError(f"Could not infer language from {path.name}")
        yield language, result
        return

    for language, n_results in result.items():
        yield str(language).upper(), n_results


def load_random_isc_rows(paths=RANDOM_SUMMARY_FILES, keep_ns=KEEP_NS) -> pd.DataFrame:
    require_files(paths)
    rows = []

    for path in paths:
        path = Path(path)
        seed = seed_from_path(path)
        result = load_pickle(path)

        for language, n_results in iter_random_blocks(path, result):
            for n, records in n_results.items():
                n = int(n)
                if n not in keep_ns:
                    continue

                for iteration, record in enumerate(records):
                    rows.append(
                        {
                            "seed": seed,
                            "iteration": iteration,
                            "language": language,
                            "n": n,
                            "participants": 2 * n,
                            "isc": mean_isc(record["isc"]),
                        }
                    )

    if not rows:
        raise ValueError("No ISC records matched the requested sample sizes")
    return pd.DataFrame(rows)


In [ ]:
df = load_random_isc_rows()

summary = (
    df.groupby(["language", "n", "participants"], as_index=False)
    .agg(
        mean_isc=("isc", "mean"),
        sd_isc=("isc", lambda values: np.nanstd(values, ddof=1)),
        n_obs=("isc", "count"),
    )
    .sort_values(["n", "language"])
)
summary["sem_isc"] = summary["sd_isc"] / np.sqrt(summary["n_obs"])
summary["ci95"] = summary.apply(
    lambda row: t.ppf(0.975, df=row["n_obs"] - 1) * row["sem_isc"]
    if row["n_obs"] > 1
    else np.nan,
    axis=1,
)

LANG_ORDER = [language for language in LANG_ORDER if language in set(summary["language"])]
ns = sorted(summary["n"].unique())
x = np.arange(len(ns))
bar_width = 0.22
offsets = (np.arange(len(LANG_ORDER)) - (len(LANG_ORDER) - 1) / 2) * bar_width

fig, ax = plt.subplots(figsize=(7.2, 4.6))

for offset, language in zip(offsets, LANG_ORDER):
    values = summary[summary["language"] == language].set_index("n").reindex(ns)
    ax.bar(
        x + offset,
        values["mean_isc"],
        width=bar_width,
        yerr=values["sd_isc"],
        label=language,
        color=LANG_COLORS[language],
        edgecolor="white",
        linewidth=1.0,
        capsize=4,
        error_kw={"elinewidth": 1.2, "ecolor": "0.25", "capthick": 1.2},
        zorder=3,
    )

ax.set_xticks(x)
ax.set_xticklabels([str(n * 2) for n in ns], fontsize=11)
ax.set_xlabel("Number of participants", fontsize=12)
ax.set_ylabel("Whole-brain ISC", fontsize=12)
ax.tick_params(axis="y", labelsize=11)

ymax = (summary["mean_isc"] + summary["sd_isc"]).max()
ax.set_ylim(0, ymax * 1.15)

ax.grid(axis="y", color="0.88", linewidth=0.8, zorder=0)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_linewidth(0.8)
ax.spines["bottom"].set_linewidth(0.8)
ax.legend(title="Language", frameon=False, fontsize=10, title_fontsize=10, loc="upper left")

plt.tight_layout()
plt.show()

summary


In [ ]:
def normalize_topbottom_result(result: dict, language: str) -> dict:
    keys = {str(key).lower() for key in result}
    if {"top", "bottom"}.issubset(keys):
        return result
    return get_case_insensitive(result, language)


def normalize_random_result(result: dict, language: str) -> dict:
    if not result:
        raise ValueError("Random ISC result is empty")

    first_key = next(iter(result))
    if isinstance(first_key, (int, np.integer)) or str(first_key).isdigit():
        return result
    return get_case_insensitive(result, language)


def load_topbottom_pair(language: str):
    language = language.upper()
    paths = TOPBOTTOM_FILES[language]
    require_files([paths["topbottom"], paths["random"]])

    topbottom = normalize_topbottom_result(load_pickle(paths["topbottom"]), language)
    random_result = normalize_random_result(load_pickle(paths["random"]), language)
    return topbottom, random_result, paths


def plot_topbottom_isc(language: str, figsize=(10, 5), show_points=True, title=None):
    res_tb, res_rand, paths = load_topbottom_pair(language)

    top_by_n = {int(n): records for n, records in get_case_insensitive(res_tb, "top").items()}
    bottom_by_n = {int(n): records for n, records in get_case_insensitive(res_tb, "bottom").items()}
    random_by_n = {int(n): records for n, records in res_rand.items()}

    n_list = [n for n in sorted(top_by_n) if n in bottom_by_n and n in random_by_n]
    if not n_list:
        raise ValueError(f"No shared n values for {language}")

    means = {"top": [], "bot": [], "rand": []}
    sds = {"top": [], "bot": [], "rand": []}
    all_vals = {"top": [], "bot": [], "rand": []}
    p_tb, p_tr, p_br = [], [], []

    for n in n_list:
        top_vals = [mean_isc(record["isc"]) for record in top_by_n[n]]
        bottom_vals = [mean_isc(record["isc"]) for record in bottom_by_n[n]]
        random_vals = [mean_isc(record["isc"]) for record in random_by_n[n]]

        all_vals["top"].append(top_vals)
        all_vals["bot"].append(bottom_vals)
        all_vals["rand"].append(random_vals)

        for label, values in zip(
            ["top", "bot", "rand"],
            [top_vals, bottom_vals, random_vals],
        ):
            means[label].append(np.mean(values))
            sds[label].append(np.std(values, ddof=1))

        p_tb.append(ttest_ind(top_vals, bottom_vals, equal_var=False).pvalue)
        p_tr.append(ttest_ind(top_vals, random_vals, equal_var=False).pvalue)
        p_br.append(ttest_ind(bottom_vals, random_vals, equal_var=False).pvalue)

    x = np.arange(len(n_list))
    width = 0.25

    fig, ax = plt.subplots(figsize=figsize)
    ax.bar(
        x - width,
        means["top"],
        width,
        yerr=sds["top"],
        label="Top",
        color=CONDITION_COLORS["top"],
        capsize=4,
    )
    ax.bar(
        x,
        means["bot"],
        width,
        yerr=sds["bot"],
        label="Bottom",
        color=CONDITION_COLORS["bot"],
        capsize=4,
    )
    ax.bar(
        x + width,
        means["rand"],
        width,
        yerr=sds["rand"],
        label="Random",
        color=CONDITION_COLORS["rand"],
        capsize=4,
    )

    if show_points:
        rng = np.random.default_rng(0)
        jitter = 0.06
        x_positions = {"top": x - width, "bot": x, "rand": x + width}
        for i in range(len(n_list)):
            for label in ["top", "bot", "rand"]:
                y = np.asarray(all_vals[label][i], dtype=float)
                xj = x_positions[label][i] + rng.uniform(-jitter, jitter, size=len(y))
                ax.scatter(
                    xj,
                    y,
                    s=18,
                    color=POINT_COLORS[label],
                    alpha=0.6,
                    linewidths=0,
                    zorder=3,
                )

    ax.set_xticks(x)
    ax.set_xticklabels([f"{n * 2}" for n in n_list])
    ax.set_xlabel("participants")
    ax.set_ylabel("ISC")
    if title:
        ax.set_title(title)
    ax.legend(frameon=False, loc="upper left")
    ax.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    plt.show()

    result = pd.DataFrame(
        {
            "n": n_list,
            "Top_mean": means["top"],
            "Top_SD": sds["top"],
            "Bottom_mean": means["bot"],
            "Bottom_SD": sds["bot"],
            "Random_mean": means["rand"],
            "Random_SD": sds["rand"],
            "p_Top_vs_Bottom": p_tb,
            "p_Top_vs_Rand": p_tr,
            "p_Bot_vs_Rand": p_br,
        }
    )

    print(f"Top/bottom file: {paths['topbottom']}")
    print(f"Random file: {paths['random']}")
    print(result.to_string(index=False, float_format="%.4f"))
    return result, res_tb, all_vals


In [ ]:
summary_en_topbottom, res_tb_en, all_vals_en = plot_topbottom_isc(
    "EN",
    figsize=(10, 5),
    show_points=True,
)

top_best_ids = {}
for n, records in res_tb_en["top"].items():
    isc_values = [mean_isc(record["isc"]) for record in records]
    best_idx = int(np.argmax(isc_values))
    top_best_ids[int(n)] = best_idx
    print(f"n={n}: best ISC={isc_values[best_idx]:.4f} idx={best_idx}")

summary_en_topbottom


In [ ]:
summary_cn_topbottom, res_tb_cn, all_vals_cn = plot_topbottom_isc(
    "CN",
    figsize=(8, 5),
    show_points=False,
    title="Chinese Group-wise ISC",
)

summary_cn_topbottom
